This script loads in the data and runs all analysis on it. Make Task= "A" if you want to test binary Real/AI images, and make Task = "B" if you want to test classifying different AI models. The final 2 cells load in the dataset in the form of pixel values, but these are optional and take a long time to load. The whole script prior to that can take between 1-2 hours to run on a GPU, which we highly recommend. Feel free to alter the number of epochs if you desire a faster or more thoroughly trained model.

In [ ]:
from datasets import load_dataset #importing all needed packages
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import torch.optim as optim
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

ds = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset") # from dataset link; loads in images

In [ ]:
ds

In [ ]:
# "A" = binary (Label_A)
# "B" = 6-class (Label_B)
TASK = "B"

In [ ]:
weights = models.ConvNeXt_Tiny_Weights.DEFAULT # pre trained weights
preprocess = weights.transforms() # gets preprocessing pipeline of the weights

In [ ]:
def make_transform(task="A"): # converts images to tensors using ConvNeXt preprocessing and sets proper labels
    def transform(batch):
        images = [img.convert("RGB") for img in batch["Image"]]
        batch["pixel_values"] = [preprocess(img) for img in images] # pixel values

        if task == "A": # binary
            batch["labels"] = batch["Label_A"]
        elif task == "B": # 6-class
            batch["labels"] = batch["Label_B"]
        else:
            raise ValueError("task must be 'A' or 'B'")

        return batch
    return transform

In [ ]:
ds_task = ds.with_transform(make_transform(TASK)) # apply make_transform preprocessing to dataset

In [ ]:
def collate_fn(batch): # combines list of images into a single batch
    pixel_values = torch.stack([item["pixel_values"] for item in batch]) # stacks image tensors
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long) # groups labels into tensors
    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
train_loader = DataLoader(
    ds_task["train"],
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
) #feeds the train dataset to the model in batches of batch size 32. Shuffled for randomness in training

val_loader = DataLoader(
    ds_task["validation"],
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
) #feeds the train dataset to the model in batches of batch size 32. NOT shuffled

test_loader = DataLoader(
    ds_task["test"],
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
) #feeds the train dataset to the model in batches of batch size 32. NOT shuffled

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # runs with GPU; significantly faster

In [ ]:
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT) # specific model type; ConvNeXt_Tiny. Faster

num_classes = 2 if TASK == "A" else 6 # adjusts num_classes for task
model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes) # adds final layer that outputs right number of classes

model = model.to(device) # applies GPU if used else CPU

In [ ]:
criterion = nn.CrossEntropyLoss() # loss metric
optimizer = optim.AdamW(model.parameters(), lr=1e-4) # optimizer

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device): # trains one epoch
    model.train() # train
    # TODO: create variables for measuring loss, number of correct preds, and number of total preds

    for batch in loader:
        inputs = batch["pixel_values"].to(device) # pixel values / inputs
        labels = batch["labels"].to(device) # labels / expected outputs

        #TODO: perform one training step (forward pass, compute loss, backpropagation, optimizer)

        #TODO: update your loss, correct, and total variables


    #TODO: return average loss and accuracy for this epoch

In [ ]:
def evaluate(model, loader, criterion, device): # similar to train_one_epoch but it does NOT update the model it just measures model performance
    model.eval() # eval mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad(): #NOT used in training, used for evaluation
        for batch in loader:
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            #TODO: run forward pass and compute loss

            #TODO: update your loss, correct, and total variables

    #TODO: return average loss and accuracy for this epoch

In [ ]:
ds_task["train"][0] #see the pixel values below for one row

In [ ]:
epochs = 5 # adjust as needed!

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device) # trains
    val_loss, val_acc = evaluate(model, val_loader, criterion, device) # evaluates

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device) # evaluate one more time on test set
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

In [ ]:
def get_preds_and_labels(model, loader, device): # runs model and collects all predicted labels and real labels
    model.eval()
    all_preds = [] # collects all predictions
    all_labels = [] # collects all true labels

    with torch.no_grad():
        for batch in loader:
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device) # gets labels

            # TODO: generate predictions and store predicted/true labels for evaluation

    return np.array(all_preds), np.array(all_labels) # returns both arrays

In [ ]:
class_names = ["Real", "AI"]
if TASK == "B":
  class_names =  ["Real", "SD21", "SDXL", "SD3", "DALLE3", "Midjourney"] # set up for confusion matrix depending on task

In [ ]:
preds, labels = get_preds_and_labels(model, test_loader, device)

In [ ]:
#TODO: Create confusion matrix to visualize results

In [ ]:
f1 = f1_score(labels, preds, average="macro")
print("Macro F1 Score:", f1)

Macro F1 Score: 0.45034807070081423
